# Module 0: 環境設定與串流測試

## 學習目標
- 認識 Google Colab 與 GPU 加速
- 學會從全球電視串流擷取即時影像
- 基礎影像處理操作（灰階、邊緣偵測、模糊）

## 事前準備
1. 執行選單 **Runtime → Change runtime type → T4 GPU**
2. 從上到下依序執行每個 Cell（按 Shift+Enter）

---
## Step 1: 安裝必要套件

In [ ]:
# 安裝 AI 視覺辨識所需的套件（約 30 秒）
!pip install ultralytics opencv-python-headless pillow easyocr yt-dlp -q
print('所有套件安裝完成！')

## Step 2: 確認 GPU 可用

In [ ]:
# 確認 Colab 有分配到 GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU 已就緒: {gpu_name}')
    print(f'GPU 記憶體: {gpu_mem:.1f} GB')
else:
    print('未偵測到 GPU！')
    print('請到選單 Runtime → Change runtime type → 選擇 T4 GPU')

## Step 3: 載入工具函式

以下是本次工作坊會用到的顯示工具，**不需要修改，直接執行即可**。

In [ ]:
# === 工具函式（直接執行，不需修改）===
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import time

# 設定 matplotlib 中文顯示
plt.rcParams['font.size'] = 12

def show_frame(frame, title='', size=(640, 360)):
    """在 Colab 中顯示一幀影像"""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb).resize(size)
    if title:
        print(title)
    display(img)

def show_compare(images, titles, size=(320, 240)):
    """並排比較多張影像"""
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 4))
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def grab_frame(source, timeout=15):
    import subprocess
    url = source
    # 'search:關鍵字' → 用 yt-dlp 找「現在正在直播」的串流（自我修復，不怕單一 ID 失效）
    if isinstance(source, str) and source.startswith('search:'):
        q = source[len('search:'):]
        try:
            url = subprocess.run(['yt-dlp', 'ytsearch12:' + q, '--match-filter', 'is_live',
                                  '--print', '%(webpage_url)s', '-I', '1:1', '--no-warnings'],
                                 capture_output=True, text=True, timeout=90).stdout.strip().split('\n')[0]
        except Exception:
            url = ''
    # YouTube 網址 → 解析成可直接讀的串流
    if url and ('youtube.com' in url or 'youtu.be' in url):
        try:
            url = subprocess.run(['yt-dlp', '-f', 'best[ext=mp4]/best', '-g', url],
                                 capture_output=True, text=True, timeout=60).stdout.strip().split('\n')[0]
        except Exception:
            url = ''
    # 國道即時影像是 MJPEG → curl 取一張 JPEG（cv2 對 MJPEG 不穩）
    if url and 'abs2mjpg' in url:
        data = subprocess.run(['curl', '-s', '--max-time', '8', url], capture_output=True).stdout
        s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
        if s >= 0 and e >= 0:
            return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    elif url:
        cap = cv2.VideoCapture(url)
        cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, timeout * 1000)
        ret, frame = cap.read(); cap.release()
        if ret:
            return frame
    # 備援：台灣國道即時影像，保證有畫面
    print('來源連線失敗，改用國道即時影像備援')
    data = subprocess.run(['curl', '-s', '--max-time', '8',
        'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001'], capture_output=True).stdout
    s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
    if s >= 0 and e >= 0:
        return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    raise ConnectionError('來源與備援皆失敗，請稍後再試')

print('工具函式載入完成！')

---
## Step 4: 串流來源清單

以下是來自全球合法公開電視串流（來源：[iptv-org](https://github.com/iptv-org/iptv)）。

每個場景都有多個備用串流，如果第一個連不上，試下一個即可。

In [ ]:
# === 全球即時串流來源 ===
# 每個場景都準備了多個備用來源，連不上就換下一個

STREAMS = {
    # === 交通（台灣國道即時影像，政府 CCTV，最穩定）===
    '國道1號 高架(車多)': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=11470',
    '國道3號': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=30001',
    '國道2號 機場': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=20000',
    # === 多樣場景（YouTube 直播，用搜尋詞自動找「現在正在直播」的，不怕單一 ID 失效）===
    '動物 (非洲 safari 直播)': 'search:african safari waterhole live animals',
    '城市人潮 (路口直播)': 'search:city street crossing live cam people',
    '港口/船隻 (直播)': 'search:live harbour port ships cam',
    '野生鳥類 (直播)': 'search:bird feeder live cam',
}

print(f'已載入 {len(STREAMS)} 個串流來源！')
print()
for name in STREAMS:
    print(f'  - {name}')

## Step 5: 抓取你的第一個即時畫面！

從上面的清單選一個頻道名稱，填入下方程式碼中。

In [ ]:
# === 動手做！===
# 把下面的頻道名稱換成你想看的
channel = '國道1號 基隆端'

# 抓取一幀
frame = grab_frame(STREAMS[channel])
show_frame(frame, title=f'即時畫面: {channel}')

### 小練習：試試不同的頻道

把 `channel = '...'` 改成其他頻道名稱，重新執行。
如果某個頻道連不上，代表該串流目前離線，換一個就好。

---
## Step 6: 基礎影像處理

電腦看到的影像其實就是一堆數字。我們來看看常見的影像處理操作。

In [ ]:
# === 影像的本質 ===
print(f'影像形狀: {frame.shape}')
print(f'  → 高度: {frame.shape[0]} 像素')
print(f'  → 寬度: {frame.shape[1]} 像素')
print(f'  → 色彩通道: {frame.shape[2]} (藍、綠、紅)')
print(f'  → 總共 {frame.shape[0] * frame.shape[1]:,} 個像素')
print(f'  → 每個像素的值: 0（黑）~ 255（白）')
print()
print(f'左上角那個像素的顏色值: {frame[0, 0]}  (藍, 綠, 紅)')

In [ ]:
# === 灰階轉換 ===
# 把彩色變成黑白，很多演算法的第一步
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# === 邊緣偵測 ===
# Canny 演算法：找出影像中物體的輪廓
edges = cv2.Canny(gray, 50, 150)

# === 模糊處理 ===
# 高斯模糊：讓影像變柔和，常用來去雜訊
blurred = cv2.GaussianBlur(frame, (21, 21), 0)

# 並排比較
show_compare(
    [frame, gray, edges, blurred],
    ['Original', 'Grayscale', 'Edge Detection', 'Gaussian Blur']
)

In [ ]:
# === 裁剪影像 ===
# 只取畫面中間的區域
h, w = frame.shape[:2]
cropped = frame[h//4 : 3*h//4, w//4 : 3*w//4]  # 中間 50% 區域

show_compare(
    [frame, cropped],
    ['Original', 'Cropped (Center 50%)']
)

---
## Step 7: 連續擷取多幀

即時串流是連續不斷的畫面。我們來擷取多幀，觀察畫面的變化。

In [ ]:
# === 從串流連續擷取 5 幀 ===
channel = '國道1號 0K+100'  # 可換成其他頻道
url = STREAMS[channel]

cap = cv2.VideoCapture(url)
frames = []

print(f'正在從 {channel} 擷取 5 幀...')
for i in range(5):
    ret, f = cap.read()
    if ret:
        frames.append(f)
        print(f'  第 {i+1} 幀 OK')
    time.sleep(1)  # 每秒抓一幀

cap.release()
print(f'擷取完成！共 {len(frames)} 幀')

In [ ]:
# === 顯示這 5 幀 ===
if frames:
    show_compare(
        frames[:5],
        [f'Frame {i+1}' for i in range(len(frames[:5]))]
    )

---
## Module 0 完成！

### 你學會了：
- Colab GPU 環境設定
- 從全球電視串流擷取即時影像
- 影像的本質（像素矩陣）
- 基礎影像處理：灰階、邊緣偵測、模糊、裁剪

### 接下來：
→ 開啟 **01_detection.ipynb**，讓 AI 來辨識畫面中的所有物件！